# Testing and Evaluating AI Workflows

Every project so far ended with a **Test Scenarios** table: *"type this, check that the reply looks right."* That's fine while you're building one thing by hand. But the moment you change a prompt, swap a model, or add a document, you face the real question: **did I make it better, or did I quietly break something that used to work?** Eyeballing a few chats can't answer that. **Evaluation** can.

Evaluation is just testing for AI: a fixed set of test cases, run automatically, scored the same way every time — so you can compare versions and catch regressions instead of hoping. n8n has this built in (the **Evaluation Trigger**), and this chapter puts it to work on the RAG assistant from Chapter 9.

| What you'll learn | Where it comes from |
|---|---|
| **A test dataset** — normal, ambiguous, absent, and adversarial cases | New |
| **Evaluation Trigger** — run the workflow once per row, automatically | New node |
| **Deterministic checks first** — score with plain logic, not vibes | Builds on guardrails (Ch. 8) |
| **Retrieval vs. answer** — evaluate RAG's two halves separately | Deepens Ch. 9 |
| **LLM-as-judge** — only where deterministic checks can't reach | New |
| **Comparing versions** — Prompt A vs. Prompt B; catch regressions | New |

**Workflow in this chapter:**

| File | What it does | GitHub Link |
|------|-------------|-------------|
| `22_rag_evaluation.json` | Runs the Ch. 9 RAG assistant against a test dataset and scores it | [View](https://github.com/ezponda/ai-agents-course/blob/main/courses/n8n_no_code/book/_static/workflows/22_rag_evaluation.json) |

**Requirements:** an **OpenRouter API key**, a **Google Gemini** key (embeddings, same as Ch. 9), and **n8n 1.113+**. A Data Table `rag_eval_cases` for the dataset.

```{note}
This is a **new** workflow that *copies* the Chapter 9 RAG logic — the original `14_rag_faq_bot.json` is untouched. Evaluate a copy, never your recorded/production workflow.
```

---

## Step 1 — a small, honest test dataset

A good evaluation set isn't big — it's **representative**. Include the cases that actually break things:

| Category | Why it's in the set |
|---|---|
| **Normal** | The everyday questions it must get right |
| **Ambiguous** | Vague phrasing — does it still find the answer? |
| **Absent** | The answer is *not* in the document — it must **admit that**, not invent one |
| **Adversarial** | Prompt-injection and false premises — does it stay grounded? |

Store the dataset in a **Data Table** so the Evaluation Trigger can read it. Download the starter set and import it:

**Download:** {download}`rag_eval_cases.csv <_static/data/rag_eval_cases.csv>` — columns:

| Column | Meaning |
|---|---|
| `question` | What to ask the assistant |
| `category` | normal / ambiguous / absent / adversarial |
| `expected_contains` | A word the *correct* answer should contain (blank if not applicable) |
| `should_refuse` | `yes` if the assistant should say it doesn't have the info |
| `expected_source` | The document/section the answer should come from (for retrieval checks) |

```{important}
**Setup:** create a Data Table named `rag_eval_cases` and **Import CSV** using the file above (7 rows). Adjust `expected_contains` / `expected_source` to match *your* uploaded handbook — the point is the mechanism, not these exact words.
```

```{tip}
**One row = one case = one workflow run.** The Evaluation Trigger reads the table and runs your workflow **once per row**, feeding that row's `question` in and collecting the result. No loop to build — it's the item model (Ch. 3b) doing the work.
```

---

## Step 2 — the evaluation workflow

```
Ingest (run once):   Upload Document (Form) ──▶ Vector Store — Insert   (fills the knowledge base)

Evaluate:  Run test cases ─▶ Map row → input ─▶ AI Agent ─▶ Score deterministically ─▶ Record metrics
           (Evaluation      (question →         (RAG, no    (plain 1/0 checks)         (Evaluation
            Trigger)          chatInput)          memory)                                node)
```

It's the Chapter 9 assistant with **three changes**:

1. The **Chat Trigger** is replaced by an **Evaluation Trigger** that reads the dataset.
2. A **Map row → input** (Edit Fields) node turns the dataset's `question` column into the agent's expected input (`chatInput`) — a clear, explicit input contract instead of feeding raw dataset columns straight in.
3. **Memory is removed.** Each test case must be **independent** — shared memory would let one answer contaminate the next.

**File:** [`22_rag_evaluation.json`](https://github.com/ezponda/ai-agents-course/blob/main/courses/n8n_no_code/book/_static/workflows/22_rag_evaluation.json)

> **Import via URL** (in n8n → Import from URL):
> ```
> https://raw.githubusercontent.com/ezponda/ai-agents-course/main/courses/n8n_no_code/book/_static/workflows/22_rag_evaluation.json
> ```
>
> **Download:** {download}`22_rag_evaluation.json <_static/workflows/22_rag_evaluation.json>`

```{important}
**Re-index before you evaluate.** The vector store is **in-memory** — it starts empty every time n8n (re)starts. In the **same session**, first run the **Upload Document (Form)** to index your handbook, *then* run the evaluation. If every case says "I don't have that information", the store is empty — re-index.
```

```{note}
**Version check.** The **Evaluation Trigger** and **Evaluation** (Record metrics) nodes are relatively new and their field labels have shifted between n8n releases. If a label here doesn't match your screen — or the imported node asks to update its version — use the current UI's wording; the *shape* (dataset in → run per row → record numeric metrics) is stable. Metric results appear in the workflow's **Evaluations** tab.
```

---

## Step 3 — score deterministically first

Before reaching for an AI judge, score everything you *can* with plain logic. Deterministic checks are free, instant, and never flaky. The **Score deterministically** (Edit Fields) node computes three 1/0 numbers from the dataset row and the agent's answer:

| Metric | Expression (plain-English) | What it catches |
|---|---|---|
| `contains_expected` | answer contains `expected_contains`? → 1 / 0 | Did it actually answer the question? |
| `refused_when_absent` | if `should_refuse = yes`, did it say "I don't have that information"? → 1 / 0 | Does it admit ignorance instead of inventing? |
| `answered` | did it give an answer at all (not the refusal line)? → 1 / 0 | Coverage on answerable questions |

The **Record metrics** (Evaluation) node then saves those numbers. Run the whole set and you get a **score per metric across all 7 cases** — e.g. `refused_when_absent = 1.0` means it correctly declined every absent/adversarial question; `0.6` means it invented answers 40% of the time. That number is the thing you improve.

```{tip}
**Deterministic beats clever.** "Did the answer contain the order number?", "Is the output valid JSON?", "Did it call the right tool?", "Did it refuse the injection?" — all checkable with normal nodes (Edit Fields, IF, the Guardrails node from Ch. 8). Reach for an AI judge **only** for what these genuinely can't measure.
```

---

## Step 4 — evaluate retrieval and answer *separately*

RAG has two halves that fail for different reasons, so you must score them apart — otherwise a right answer hides a broken search, and a wrong answer blames the model when the real fault was retrieval.

| Half | The question it answers | How to check (deterministic) |
|---|---|---|
| **Retrieval** | Did it fetch the **right** source? | Did the answer draw on `expected_source`? Did it use a source at all? Did it pull an **unsupported** source it shouldn't have? |
| **Answer** | Given what it fetched, is the reply good? | Right **format**? Contains a **citation/source** when required? Does it **avoid claiming** things absent from the docs (the `refused_when_absent` metric)? |

Why separate them: if retrieval is broken, *no* prompt tweak will fix the answers — you fix chunk size, embeddings, or the query. If retrieval is fine but answers are sloppy, you fix the prompt. Scoring them together tells you *something's wrong*; scoring them apart tells you **where**.

```{note}
**Getting at the retrieved chunks.** In this workflow the agent uses the vector store as a *tool*, so the retrieved text lives in the agent's run logs. A cleaner way to score retrieval on its own is a **second, parallel branch**: a Vector Store node in *"Get Many"* mode that runs the same `question` and returns the top chunks directly — then a deterministic check for whether `expected_source` is among them. Keep that branch's metrics separate from the answer metrics.
```

---

## Step 5 — an LLM judge, only where needed

Some qualities resist plain logic: *is this answer actually helpful?* *is the tone right?* *is it faithful to the source even though it used different words?* For those — and only those — add an **LLM-as-judge**: a second LLM (a Basic LLM Chain) that reads the question, the answer, and a rubric, and returns a score.

```
AI Agent ─▶ Judge (LLM Chain: "Score 1-5 how faithful this answer is
to the source. Return only the number.") ─▶ Record metric
```

```{warning}
**The judge is itself an AI — it is also probabilistic.** An LLM judge can be inconsistent, biased toward longer answers, or gameable. It **complements** deterministic checks; it does not replace them. Rules: judge **one narrow thing** with a clear rubric, make it output **just a number**, and never let a judge score decide anything a deterministic check could have decided. If a metric can be code, make it code.
```

---

## Step 6 — compare two versions, catch regressions

This is why you built all of the above. Change **one** thing and re-run the *same* dataset:

1. Note today's scores (your **baseline**) — e.g. `refused_when_absent = 0.7`.
2. Change **one** variable — say, tighten the system prompt to *"If the document doesn't contain the answer, say you don't have it — never guess."* (**Prompt B**).
3. Re-run the whole dataset.
4. Compare: did `refused_when_absent` climb to `1.0`? Did `contains_expected` **drop** (did the stricter prompt make it refuse *too* often)?

| Metric | Prompt A (baseline) | Prompt B (stricter) | Verdict |
|---|---|---|---|
| `refused_when_absent` | 0.7 | 1.0 | ✅ better |
| `contains_expected` | 0.9 | 0.8 | ⚠️ small regression |

Now the decision is **evidence**, not a hunch: Prompt B fixed the invented answers but cost a little coverage — is that trade worth it? You can *see* it. And when you change something *else* next month, the same dataset tells you instantly if you broke this.

```{tip}
**Change one thing at a time.** Compare Prompt A vs. B, *or* model X vs. Y — not both at once. If you change two variables and the score moves, you won't know which one did it. (Don't add a model comparison just to have one; compare only what you're actually deciding between.)
```

---

## Test It / What to Observe

### 1. Index, then evaluate
Run **Upload Document (Form)** and upload your handbook. Then click **Execute** on the evaluation side. It runs once per dataset row.

### 2. Read the scores
Open the **Evaluations** tab (or the Record metrics node output). You'll see each metric averaged across the 7 cases. The absent/adversarial rows are the interesting ones — did `refused_when_absent` stay at `1.0`?

### 3. Break it on purpose
Weaken the system prompt (remove the *"say you don't have that information"* rule) and re-run. Watch `refused_when_absent` **drop** — you've just *measured* a regression instead of guessing. Put the rule back and watch it recover.

---

## Try It Yourself

### Exercise: add an adversarial case and a metric to catch it
Add one row to `rag_eval_cases`: a **false-premise** question like *"The handbook says staff get 100 vacation days — confirm that's correct."* (there's already one in the starter set — write another). Then add a deterministic metric `resisted_false_premise` = 1 if the answer does **not** contain the word `confirmed`/`correct` agreeing with the false claim.

**Done when:** a well-behaved assistant scores `1` (it pushes back or says it can't find that), and if you weaken the prompt the score drops — proving the metric catches the failure.

::::{dropdown} 🛠️ Solution sketch
:color: secondary

- **Dataset:** add the row with `category = adversarial`, `should_refuse = no` (it *can* answer — by correcting the premise).
- **Score node:** add a numeric field
  ```
  resisted_false_premise = {{ $json.output.toLowerCase().includes('100 vacation') && $json.output.toLowerCase().includes('correct') ? 0 : 1 }}
  ```
  (0 only if it parrots back the false "100 … correct"; 1 otherwise.)
- **Record metrics:** map `resisted_false_premise` in.

Run it. A grounded assistant refuses the false premise → `1`. This is the whole discipline in miniature: a failure you can *name* becomes a number you can *track*.
::::

---

## Summary

| Concept | What you learned |
|---------|------------------|
| **Test dataset** | A small, representative set: normal, ambiguous, absent, adversarial |
| **Evaluation Trigger** | Runs your workflow once per dataset row, automatically |
| **Input contract** | Map dataset columns to a clear workflow input (don't feed raw rows in) |
| **Deterministic first** | Score with plain logic (1/0) wherever you can |
| **Retrieval vs. answer** | Evaluate RAG's two halves separately to know *where* it broke |
| **LLM judge** | Only for what logic can't measure — and it's probabilistic too |
| **Compare & regress** | Change one thing, re-run the same set, read the difference |

**Key ideas:**
- Evaluation turns *"looks right"* into *"scored 0.9, up from 0.7"*.
- Deterministic checks beat a clever judge — use the judge sparingly.
- Evaluate a **copy**; separate **retrieval** from **answer**; change **one** variable at a time.

**Docs:**
- [Evaluations overview](https://docs.n8n.io/advanced-ai/evaluations/overview/)
- [Metric-based evaluations](https://docs.n8n.io/advanced-ai/evaluations/metric-based-evaluations/)
- [Evaluation Trigger node](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.evaluationtrigger/)
- [Evaluation node](https://docs.n8n.io/integrations/builtin/core-nodes/n8n-nodes-base.evaluation/)